In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np # For plotting and meshgrid
from mpl_toolkits.mplot3d import Axes3D # For 3D plotting
import opt_einsum as oe
from itertools import combinations, product, permutations


DEVICE = torch.device('cuda:3') if torch.cuda.is_available() else torch.device('cpu')

# (Optional) Set a seed for reproducibility
# torch.manual_seed(42)
# if torch.cuda.is_available():
#   torch.cuda.manual_seed_all(42)

def construct_model(
  arch,
  train_embed_layer,
  act,
  width,
  hid_n_layers,
):
  # arch : "MLP", "ResNet", "DiffNet-Add", "DiffNet-Mul"
  # train_embed_layer : bool, whether to train the first layer
  # act : "tanh", "sin", "relu", "gelu"
  # width : int, number of hidden units in each layer
  # hid_n_layers : int, number of hidden layers
  if arch == "MLP":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=False
    )
  elif arch == "ResNet":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=True
    )
  elif arch == "DiffNet-Add":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='add'
    )
  elif arch == "DiffNet-Mul":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='mul'
    )
  elif arch.startswith("DiffNet-Const"):
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='const',
      scale_const=float(arch.split('-')[-1]) if '-' in arch else 1.0
    )
  else:
    raise ValueError(f"Unknown architecture: {arch}")

class MLP(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    residual=False
  ):
    super(MLP, self).__init__()
    self.train_embed_layer = train_embed_layer
    self.width = width
    self.hid_n_layers = hid_n_layers
    self.residual = residual

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin

    # Only first activation is sin, and the rest of the network uses given activation
    
    self.embed_layer = nn.Linear(2, width, bias=True)
    if not self.train_embed_layer:
      self.embed_layer.weight.requires_grad = False
      self.embed_layer.bias.requires_grad = False
    
    # hid_n_layers-1 hidden layers
    self.hidden_layers = nn.ModuleList()
    for _ in range(hid_n_layers - 1):
      self.hidden_layers.append(nn.Linear(width, width, bias=True))
    
    # Output layer
    self.output_layer = nn.Linear(width, 1, bias=True)
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    # Concatenate the two scalar inputs
    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    # Pass through the embedding layer
    x = self.embed_layer(combined_input)
    x = torch.sin(x)  # First activation is sin
    for layer in self.hidden_layers:
      h = layer(x)
      if self.residual:
        x = self.act(h) + x
      else:
        x = self.act(h)
    # Output layer
    x = self.output_layer(x)
    return x  # Ensure output is in [0, 1]

class DiffNet(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    scale_type='add',  # 'add' or 'mul'
    scale_const=1.0,
  ):
    super(DiffNet, self).__init__()

    # we have hid_n_layer + 1 embed layers
    self.begin_embed_layer = nn.Linear(2, width, bias=True)
    if not train_embed_layer:
      self.begin_embed_layer.weight.requires_grad = False
      self.begin_embed_layer.bias.requires_grad = False
    
    self.embed_layers = nn.ModuleList()
    self.hidden_layers = nn.ModuleList()
    for i in range(hid_n_layers):
      embed_layer = nn.Linear(2, width, bias=True)
      # if scale_type is add, we multiply the weight / bias by (i + 2)
      # if scale_type is mul, we multiply the weight / bias by 2^(i+1)
      if scale_type == 'add':
        embed_layer.weight.data *= (i + 2)
        embed_layer.bias.data *= (i + 2)
      elif scale_type == 'mul':
        embed_layer.weight.data *= 2**(i + 1)
        embed_layer.bias.data *= 2**(i + 1)
      elif scale_type == 'const' and scale_const != 1.0:
        embed_layer.weight.data *= scale_const
        embed_layer.bias.data *= scale_const
      
      if not train_embed_layer:
        embed_layer.weight.requires_grad = False
        embed_layer.bias.requires_grad = False
      self.embed_layers.append(embed_layer)

      # hidden layers are 2*width -> width
      self.hidden_layers.append(nn.Linear(2 * width, width, bias=True))
    
    # Output layer
    self.output_layer = nn.Linear(width, 1, bias=True)

    # act
    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin
  
  def print_embed_var(self):
    # for each embed layer (both begin and the list)
    # 1. Get L2 norm of weights' data, which will give vector of length width
    # 2. Then take mean of that vector
    # 3. print for each layer
    print(f"Begin: {torch.mean(self.begin_embed_layer.weight.data.norm(dim=1)).item()}")
    for i, embed_layer in enumerate(self.embed_layers):
      print(f"Layer {i+1}: {torch.mean(embed_layer.weight.data.norm(dim=1)).item()}")
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    # Concatenate the two scalar inputs
    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    # Pass through the embedding layer
    x = self.begin_embed_layer(combined_input)
    x = torch.sin(x)  # First activation is sin

    for embed_layer, hidden_layer in zip(self.embed_layers, self.hidden_layers):
      h = embed_layer(combined_input)
      h = torch.sin(h)

      # Prepare input for hidden layers
      z = torch.cat((x, h), dim=-1)  # Concatenate with original inputs
      z = hidden_layer(z)
      x = self.act(z) + x

    # Output layer
    x = self.output_layer(x)
    return x

In [ ]:
import torch
import numpy as np

class ConstrainedSigmoidProduct(torch.autograd.Function):
  """
  A custom autograd function that computes the constrained sigmoid product.
  """

  @staticmethod
  def forward(ctx, y, p, lb, num_iterations=200, tolerance=1e-7, mode='newton-rhapson'):
    # Note: Default tolerance tightened to 1e-12 for float64 gradcheck stability
    
    # 1. Heuristic initialisation
    k = y.shape[1]
    # Protect against numerical issues with p=0 or p=1
    p_clamped = np.clip(p, 1e-6, 1.0 - 1e-6)
    p_root = np.power(p_clamped, 1.0 / k)
    logit_p_root = np.log(p_root / (1.0 - p_root))
    
    # Initial guess: assuming y is centered, c aligns the mean logit
    init_c = logit_p_root - torch.mean(y)
    
    # Global Minimax range for clipping
    y_min = torch.min(y)
    y_max = torch.max(y)
    minimax_range = y_max - y_min

    if mode == 'newton-rhapson':
      c = torch.full((1,), init_c.item(), device=y.device, dtype=y.dtype)

      for i in range(num_iterations):
        s = torch.sigmoid(y + c)
        
        # P_i = prod_j S_ij
        p_prod = torch.prod(s, dim=1, keepdim=True)
        
        # f(c) = mean(P) - p
        f_c = torch.mean(p_prod) - p
        
        if torch.abs(f_c) < tolerance:
          break

        # f'(c) calculation
        # sum_term = sum_j (1 - S_ij)
        term_sum = torch.sum(1.0 - s, dim=1, keepdim=True)
        f_prime_c = torch.mean(p_prod * term_sum)
        
        # Update with minimax clipping
        # Use a smaller epsilon (1e-12) for float64 compatibility
        step = f_c / (f_prime_c + 1e-7)
        
        # Dynamic clipping bound
        limit = (minimax_range + 1.0) / (i + 1)
        step = torch.clamp(step, min=-limit, max=limit)
        
        c = c - step
        c = torch.clamp(c, min=logit_p_root - y_max, max=logit_p_root - y_min)
      c_final = c

    elif mode == 'bisection':
      # Expand bounds dynamically based on y distribution. 
      # Renamed lb to low to avoid shadowing the parameter.
      low = torch.tensor([logit_p_root - y_max - 10.0], device=y.device, dtype=y.dtype)
      high = torch.tensor([logit_p_root - y_min + 10.0], device=y.device, dtype=y.dtype)

      for _ in range(num_iterations):
        midpoint = (low + high) / 2.0
        s = torch.sigmoid(y + midpoint)
        p_prod = torch.prod(s, dim=1)
        f_mid = torch.mean(p_prod) - p
        
        if torch.abs(f_mid) < tolerance:
          c_final = midpoint
          break
        
        if f_mid < 0:
          low = midpoint
        else:
          high = midpoint  
      c_final = (low + high) / 2.0

    else:
      raise ValueError("Invalid mode")
    
    ctx.save_for_backward(y, c_final)
    return c_final

  @staticmethod
  def backward(ctx, grad_output):
    y, c = ctx.saved_tensors
    
      
    # Recompute forward pass components
    s = torch.sigmoid(y + c)
    p_prod = torch.prod(s, dim=1, keepdim=True)
    
    # Denominator: Sum over batch of (P_i * sum(1-s))
    sum_1_minus_s = torch.sum(1.0 - s, dim=1, keepdim=True)
    dp_dc = p_prod * sum_1_minus_s
    
    # Stability epsilon must be small to not skew gradients in double precision
    denominator = torch.sum(dp_dc) + 1e-7
    
    # Numerator: P_i * (1 - s)
    numerator = p_prod * (1.0 - s)
    
    # Implicit gradient
    dc_dy = -numerator / denominator
    
    # Chain rule
    grad_y = grad_output * dc_dy
    
    # 6 returns for 6 input arguments (y, p, lb, num_iterations, tolerance, mode)
    return grad_y, None, None, None, None, None

constrained_sigmoid_product = ConstrainedSigmoidProduct.apply

In [ ]:
import math

# Borrowed from https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup/blob/master/cosine_annealing_warmup/scheduler.py

class CosineAnnealingWarmupRestarts(torch.optim.lr_scheduler._LRScheduler):
  
  def __init__(self,
    optimizer: torch.optim.Optimizer,
    first_cycle_steps: int,
    cycle_mult: float = 1.,
    max_lr: float = 0.1,
    min_lr: float = 0.001,
    warmup_steps: int = 0,
    gamma: float = 1.,
    last_epoch: int = -1):

    assert warmup_steps < first_cycle_steps
     
    self.first_cycle_steps = first_cycle_steps # first cycle step size
    self.cycle_mult = cycle_mult # cycle steps magnification
    self.base_max_lr = max_lr # first max learning rate
    self.max_lr = max_lr # max learning rate in the current cycle
    self.min_lr = min_lr # min learning rate
    self.warmup_steps = warmup_steps # warmup step size
    self.gamma = gamma # decrease rate of max learning rate by cycle
        
    self.cur_cycle_steps = first_cycle_steps # first cycle step size
    self.cycle = 0 # cycle count
    self.step_in_cycle = last_epoch # step size of the current cycle
    
    super(CosineAnnealingWarmupRestarts, self).__init__(optimizer, last_epoch)
    
    self.init_lr()

  def init_lr(self):
    self.base_lrs = []
    for param_group in self.optimizer.param_groups:
      param_group['lr'] = self.min_lr
      self.base_lrs.append(self.min_lr)
  
  def get_lr(self):
    if self.step_in_cycle == -1:
      return self.base_lrs
    elif self.step_in_cycle < self.warmup_steps:
      return [(self.max_lr - base_lr) * self.step_in_cycle / self.warmup_steps + base_lr for base_lr in self.base_lrs]
    else:
      return [base_lr + \
              (self.max_lr - base_lr) \
              * (1 + math.cos(math.pi * (self.step_in_cycle-self.warmup_steps) \
                              / (self.cur_cycle_steps - self.warmup_steps))) / 2
              for base_lr in self.base_lrs]
  
  def step(self, epoch=None):
    if epoch is None:
      epoch = self.last_epoch + 1
      self.step_in_cycle = self.step_in_cycle + 1
      if self.step_in_cycle >= self.cur_cycle_steps:
        self.cycle += 1
        self.step_in_cycle = self.step_in_cycle - self.cur_cycle_steps
        self.cur_cycle_steps = int((self.cur_cycle_steps - self.warmup_steps) * self.cycle_mult) + self.warmup_steps
      else:
        if epoch >= self.first_cycle_steps:
          if self.cycle_mult == 1.:
            self.step_in_cycle = epoch % self.first_cycle_steps
            self.cycle = epoch // self.first_cycle_steps
          else:
            n = int(math.log((epoch / self.first_cycle_steps * (self.cycle_mult - 1) + 1), self.cycle_mult))
            self.cycle = n
            self.step_in_cycle = epoch - int(self.first_cycle_steps * (self.cycle_mult ** n - 1) / (self.cycle_mult - 1))
            self.cur_cycle_steps = self.first_cycle_steps * self.cycle_mult ** (n)
        else:
          self.cur_cycle_steps = self.first_cycle_steps
          self.step_in_cycle = epoch
                
      self.max_lr = self.base_max_lr * (self.gamma**self.cycle)
      self.last_epoch = math.floor(epoch)
      for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
        param_group['lr'] = lr

In [ ]:
from itertools import permutations
# Let tau be nice graph parameter. 
# Consider G_n ~ G(n, p), Erdos-Renyi random graph with edge density p. 
# We find graphon W*, such that 
# min h_p(W) w.r.t. t(H, W) >= r.
# Then, the conditional distribution 
# G_N | t(H, G_N) >= r
# is very close to the optimal graphon W*.

# So we condition on t(H, W) >= r, 

def entropy(logit, density):
  # Ensure input is evaluated as a probability
  p = torch.sigmoid(logit)
  
  # Use logsigmoid for stable calculation of log(p) and log(1-p)
  log_p = F.logsigmoid(logit)
  log_1_m_p = F.logsigmoid(-logit)
  
  # Calculate logarithms of the scalar density
  log_d = math.log(density)
  log_1_m_d = math.log(1.0 - density)
  
  # Compute the relative entropy (KL divergence) as specified
  kl_div = p * (log_p - log_d) + (1.0 - p) * (log_1_m_p - log_1_m_d)
  
  return torch.mean(kl_div)

pair_to_ind = dict()

for i in range(3):
  for j in range(i + 1, 3):
    pair_to_ind[(i, j)] = len(pair_to_ind)

c3_indices = []

for perm in permutations(range(1, 3)):
  if perm[0] < perm[1]:
    new_perm = (0,) + perm
    
    curr_indices = [] 
    for i in range(3):
      first_item = new_perm[i]
      second_item = new_perm[(i + 1) % 3]
      pair = (min(first_item, second_item), max(first_item, second_item))
      curr_indices.append(pair_to_ind[pair])
    c3_indices.append(curr_indices)

c3_indices = torch.tensor(c3_indices, device=DEVICE)  # Shape: [num_permutations, 3]

def compute_objective(model, num_samples, lb, graph_density_target, edge_density_target, device):
  points = torch.rand(num_samples, 3, device=device)
  
  indices = torch.combinations(torch.arange(3, device=device), r=2)

  pairs_first = points[:, indices[:, 0]]
  pairs_second = points[:, indices[:, 1]]

  new_x = torch.minimum(pairs_first, pairs_second)
  new_y = torch.maximum(pairs_first, pairs_second)

  # Use model output directly as the logit space
  logits = model(new_x, new_y).squeeze(-1) # [num_samples, 6]

  # Extract edges for C3 and reshape
  graph_density_logits = logits[:, c3_indices].reshape(-1, 3)

  # Calculate the necessary shift to satisfy the constraint
  c_shift = constrained_sigmoid_product(graph_density_logits, graph_density_target, lb)

  # Apply the shift in logit space
  constrained_logits = logits + c_shift

  # Entropy expects logits
  obj = entropy(constrained_logits.flatten(), edge_density_target)

  # Calculate the final graph density metric using the shifted logits
  graph_density = torch.mean(torch.prod(torch.sigmoid(graph_density_logits + c_shift), dim=-1))

  if abs(graph_density.item() - graph_density_target) > 1e-3:
    print(f"Warning: Large shift value: {c_shift.item():.4f}, logits stats: mean={logits.mean().item():.4f}, std={logits.std().item():.4f}, min={logits.min().item():.4f}, max={logits.max().item():.4f}")
    print(f"Graph density: {graph_density.item():.6f}")
    raise ValueError("Abnormally large shift value, likely due to numerical instability. Check the warning message for details.")

  return obj, c_shift.item(), graph_density.item()

In [ ]:
def train_network(
    edge_density_target,
    graph_density_target,
    total_epochs=500000,
    learning_rate=1e-4,
    mc_samples=128,
    arch='DiffNet-Mul',
    act='gelu',
    network_width=64,
    network_depth=3,
    print_interval=10000,
    lb=True,
  ):
  device = DEVICE
  print(f"Using device: {device}")

  graphedge_density_target = graph_density_target ** (1/3)

  entropy = graphedge_density_target * math.log(graphedge_density_target / edge_density_target) + \
            (1 - graphedge_density_target) * math.log((1 - graphedge_density_target) / (1 - edge_density_target))

  core_model_instance = construct_model(
    arch=arch,
    train_embed_layer=True,
    act=act,
    width=network_width,
    hid_n_layers=network_depth
  ).to(device)

  print(next(core_model_instance.parameters()).device)

  # Initialize the optimizer
  optimizer = optim.Adam(core_model_instance.parameters(), lr=learning_rate)
  scheduler = CosineAnnealingWarmupRestarts(
    optimizer=optimizer,
    first_cycle_steps=total_epochs // 5,
    max_lr = learning_rate,
    min_lr = learning_rate / 100,
    warmup_steps=total_epochs // 25,
    gamma=1,
  )

  print("Starting training with")
  print(f"Network: {arch}, Activation: {act}, Width: {network_width}, Depth: {network_depth}")
  print(f"Learning Rate: {learning_rate}, MC Samples: {mc_samples}")
  print(f"Edge Density Target: {edge_density_target}, Graph Density Target: {graphedge_density_target}")

  for epoch in range(total_epochs):
    core_model_instance.train()
    optimizer.zero_grad()

    objective_exp, c, triangle_density = compute_objective(
      model=core_model_instance,
      num_samples=mc_samples,
      lb=lb,
      graph_density_target=graph_density_target,
      edge_density_target=edge_density_target,
      device=device
    )

    loss = objective_exp
    loss.backward()

    optimizer.step()
    scheduler.step()

    if (epoch + 1) % print_interval == 0:
      print(f"Epoch {epoch + 1}/{total_epochs}, Objective: {objective_exp.item():.6f} / {entropy:.6f}, Triangle Density: {triangle_density:.6f}, c: {c:.6f}")
      
  return core_model_instance

def visualise_learnt_function_f_2d(model, device, graph_density_target, graph_density_samples, lb, resolution=50):
  model.eval()

  points = torch.rand(graph_density_samples, 3, device=device)
  indices = torch.combinations(torch.arange(3, device=device), r=2)

  pairs_first = points[:, indices[:, 0]]
  pairs_second = points[:, indices[:, 1]]

  new_x = torch.minimum(pairs_first, pairs_second)
  new_y = torch.maximum(pairs_first, pairs_second)

  # Direct logit
  logits = model(new_x, new_y).squeeze(-1) 

  graph_density_logits = logits[:, c3_indices].reshape(-1, 3)

  c_shift = constrained_sigmoid_product(graph_density_logits, graph_density_target, lb)

  x_vals = (torch.arange(resolution) + 0.5) / resolution
  y_vals = (torch.arange(resolution) + 0.5) / resolution

  X, Y = torch.meshgrid(x_vals, y_vals, indexing='ij')

  grid_tensor_x = torch.minimum(X, Y).flatten().to(device)
  grid_tensor_y = torch.maximum(X, Y).flatten().to(device)

  grid_logits = model(grid_tensor_x, grid_tensor_y).squeeze(-1)

  # Apply shift and sigmoid for final density visualisation
  Z = torch.sigmoid(grid_logits + c_shift).reshape(resolution, resolution).cpu().detach().numpy()

  fig, ax = plt.subplots(figsize=(8, 7))

  im = ax.imshow(Z, cmap='viridis', origin='lower', extent=[0, 1, 0, 1], vmin=0, vmax=1)

  ax.set_xlabel('x')
  ax.set_ylabel('y')

  ax.set_title(f't(K3, W) {">" if lb else "<"}= {graph_density_target:.3f}')

  fig.colorbar(im, ax=ax, label='W(x, y)', shrink=0.8)

  return fig

In [ ]:
import os
import gc

DEBUG = False

if __name__ == '__main__':
  P_R_TARGET_LIST = [(0.1, 0.5), (0.1, 0.4), (0.1, 0.6),]
  LB = True
  
  NUM_EPOCHS = 10000 if not DEBUG else 100
  lr_list = [1e-3, 1e-4]

  MC_SAMPLES = 2 ** 17

  NETWORK_WIDTH = 256
  NETWORK_DEPTH = 5

  ARCH = 'DiffNet-Mul'
  train_embed_layer = True
  act_list = ['gelu', 'sin']

  for p_r_target, lr, act in product(P_R_TARGET_LIST, lr_list, act_list):
    print(f"\nTraining with p_target={p_r_target[0]}, r_target={p_r_target[1]}, lr={lr}, act={act}\n")
    model = train_network(
      edge_density_target=p_r_target[0],
      graph_density_target=p_r_target[1] ** 3,
      total_epochs=NUM_EPOCHS,
      learning_rate=lr,
      mc_samples=MC_SAMPLES,
      arch=ARCH,
      act=act,
      network_width=NETWORK_WIDTH,
      network_depth=NETWORK_DEPTH,
      print_interval=NUM_EPOCHS // 20,
      lb=LB,
    )

    # Save the model
    torch.save(model.state_dict(), f"best_model_le_k3_{round(p_r_target[0] * 100)}_{round(p_r_target[1] * 100)}_{lr}_{act}.pth")
  
    current_device = next(model.parameters()).device
    model.eval()

    print("\n--- Final Evaluation ---")

    with torch.no_grad():
      objective_exp, shift, k3_density = compute_objective(
        model=model,
        num_samples=MC_SAMPLES * 4, 
        lb=LB,
        graph_density_target=p_r_target[1] ** 3,
        edge_density_target=p_r_target[0],
        device=current_device
      )
      print(f"Estimated Objective: {objective_exp.item():.6f}")
      print(f"Estimated K3 Density: {k3_density:.6f}")

      x_vals = (torch.arange(256) + 0.5) / 256
      y_vals = (torch.arange(256) + 0.5) / 256

      X, Y = torch.meshgrid(x_vals, y_vals, indexing='ij')

      grid_tensor_x = torch.minimum(X, Y).flatten().to(DEVICE)
      grid_tensor_y = torch.maximum(X, Y).flatten().to(DEVICE)

      # Direct logits without the logcosh bloat
      grid_logits = model(grid_tensor_x, grid_tensor_y).squeeze(-1)
      
      objective_exp = entropy(grid_logits + shift, p_r_target[0])

      Z = torch.sigmoid(grid_logits + shift).reshape(256, 256).cpu().detach().numpy()

      k3_density = np.trace(np.linalg.matrix_power(Z / 256, 3))

      print(f"Discretised Objective: {objective_exp.item():.6f}")
      print(f"Discretised K3 Density: {k3_density:.6f}")

      os.makedirs("results", exist_ok=True)
      np.save(f"results/output_le_k3_{round(p_r_target[0] * 100)}_{round(p_r_target[1] * 100)}_{lr}_{act}.npy", Z)

      del x_vals, y_vals, X, Y, grid_tensor_x, grid_tensor_y, grid_logits, Z

      fig = visualise_learnt_function_f_2d(
        model=model,
        device=current_device,
        graph_density_target=p_r_target[1] ** 3,
        graph_density_samples=MC_SAMPLES * 4, # This is sample number, not related to cycle length
        lb=LB,
        resolution=256
      )

      fig.savefig(f"results/visualisation_ge_k3_{round(p_r_target[0] * 100)}_{round(p_r_target[1] * 100)}_{lr}_{act}.png", bbox_inches='tight')

      plt.close(fig)

      del model
      gc.collect()
      torch.cuda.empty_cache()